# Module 2 - Data Cleaning and Transformation

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession \
    .builder \
        .appName('Databricks_capstone') \
            .getOrCreate()

In [0]:
storage_account="mystoacckad"
application_id="14a14259-70ba-4c26-a136-262468ab64da"
directory_id="26af9d76-35fe-404a-b312-869c37aec9c7"
container_name="fileshare"

service_credential = dbutils.secrets.get(scope="Secrete-scope-databricks1", key="app-reg-secrets1")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               f"org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

In [0]:
folder_path=f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
csv=[file_info.name for file_info in dbutils.fs.ls(folder_path) if (file_info.name).endswith('.csv')]

df_customers_dataset=spark.read.csv(folder_path+csv[0],header=True,inferSchema=True)
df_geolocation_dataset=spark.read.csv(folder_path+csv[1],header=True,inferSchema=True)
df_order_items_dataset=spark.read.csv(folder_path+csv[2],header=True,inferSchema=True)
df_order_payments_dataset=spark.read.csv(folder_path+csv[3],header=True,inferSchema=True)
df_order_reviews_dataset=spark.read.csv(folder_path+csv[4],header=True,inferSchema=True)
df_orders_dataset=spark.read.csv(folder_path+csv[5],header=True,inferSchema=True)
df_products_dataset=spark.read.csv(folder_path+csv[6],header=True,inferSchema=True)
df_sellers_dataset=spark.read.csv(folder_path+csv[7],header=True,inferSchema=True)
df_product_category_name_translation=spark.read.csv(folder_path+csv[8],header=True,inferSchema=True)

In [0]:
from pyspark.sql.functions import *
def missing_values(df, df_name):
    print(f"Missing values in {df_name}: ")
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [0]:
missing_values(df_customers_dataset,'Customers')		

In [0]:
missing_values(df_geolocation_dataset,'Geolocation')

In [0]:
missing_values(df_order_items_dataset, 'Order Items')

In [0]:
missing_values(df_order_payments_dataset, 'Order Payments')	

In [0]:
missing_values(df_order_reviews_dataset, 'Order Reviews')	

In [0]:
missing_values(df_orders_dataset, 'Orders')			

In [0]:
missing_values(df_products_dataset, 'Products')

In [0]:
missing_values(df_sellers_dataset, 'Sellers')	

In [0]:
missing_values(df_product_category_name_translation, 'Product Category name Translation')

## Handling Missing values
### 1. Drop Missing Values (for non critical columns)
### 2. Fill Missing Values (for numerical values)
### 3. Impute Missing Values (for continous data)

In [0]:
orders_df_cleaned=df_orders_dataset.na.drop(subset=['order_id','customer_id','order_status'])
orders_df_cleaned.show()

In [0]:
orders_df_cleaned=orders_df_cleaned.fillna({'order_delivered_customer_date':'9999-12-31'})
missing_values(orders_df_cleaned, 'Orders')	

## Impute Missing Values

In [0]:
payments_df_with_null=df_order_payments_dataset.withColumn('payment_value',when(col('payment_value')!=99.33, col('payment_value')).otherwise(lit(None)))

payments_df_with_null.show()

In [0]:
from pyspark.ml.feature import Imputer
imputer=Imputer(inputCols=['payment_value'],outputCols=['payment_value_imputed']).setStrategy('median')
payments_df_cleaned=imputer.fit(payments_df_with_null).transform(payments_df_with_null)
payments_df_cleaned.show()

## Standardizing the Format

In [0]:
def print_schema(df,df_name):
  print(f"Schema of {df_name} :")
  print(df.printSchema())

In [0]:
print_schema(df_orders_dataset, 'Orders')	

In [0]:
print_schema(df_customers_dataset,'Customers')		

In [0]:
orders_df_cleaned=orders_df_cleaned.withColumn('order_purchase_timestamp',to_date(col('order_purchase_timestamp')))

In [0]:
orders_df_cleaned.show()

In [0]:
payments_df_cleaned.show()

In [0]:
payments_df_cleaned=payments_df_cleaned.withColumn('payment_type', \
                                                   when(col('payment_type')=='boleto', 'Bank Transfer') \
                                                   .when(col('payment_type')=='credit_card','Credit Card')\
                                                   .when(col('payment_type')=='debit_card','Debit Card').otherwise('Other'))

In [0]:
payments_df_cleaned.show()

In [0]:
print_schema(df_customers_dataset,'Customers')

In [0]:
customers_df_cleaned=df_customers_dataset.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))
print_schema(customers_df_cleaned,'Customers')

### Remove Duplicates

In [0]:
#Removing duplicates from customers
customers_df_cleaned=customers_df_cleaned.dropDuplicates(['customer_id'])

In [0]:
order_with_details_df=orders_df_cleaned.join(df_order_items_dataset, 'order_id','left') \
    .join(payments_df_cleaned,'order_id','left') \
        .join(customers_df_cleaned,'customer_id','left') 

In [0]:
order_with_details_df.show()

In [0]:
order_with_details_df.printSchema()

In [0]:
#delivery time calculation
order_with_details_df=order_with_details_df\
    .withColumn('actual_delivery_time', \
        datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [0]:
order_with_details_df.show()

In [0]:
#Advance Transformation
quantiles=df_order_items_dataset.approxQuantile('price',[0.01,0.99],0.0)
low_cutoff,high_cutoff=quantiles[0],quantiles[1]

In [0]:
df_order_items_dataset.select('price').summary().show()

In [0]:
order_item_df_clean= df_order_items_dataset.filter((col('price')<=high_cutoff) & (col('price')>=low_cutoff))

In [0]:
order_item_df_clean.show()

In [0]:
payments_df_cleaned.select('payment_installments').summary().show()

In [0]:
products_df_clean=df_products_dataset \
    .withColumn('product_size_category', \
        when(col('product_weight_g')<500,'Small') \
            .when(col('product_weight_g').between(500,2000),'Medium') \
                .otherwise('Large')
)

In [0]:
products_df_clean.show()

In [0]:
#calculate total revenue per seller
total_revenue_seller=order_item_df_clean.groupBy('seller_id').agg(sum(col('price')).alias('total_revenue')).orderBy(col('seller_id').asc())

In [0]:
total_revenue_seller.show(truncate=False)

In [0]:

adls_path=f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
order_with_details_df.write.mode('overwrite').parquet(adls_path+'transformed_data/order_details')

In [0]:
dbutils.fs.ls('abfss://fileshare@mystoacckad.dfs.core.windows.net/transformed_data/')

In [0]:
# Read parquet folder
df = spark.read.parquet(adls_path+'transformed_data/order_details')

# Register temporary table for SQL queries
df.createOrReplaceTempView("order_details_temp")

# Now query
spark.sql("SELECT * FROM order_details_temp LIMIT 10").show()
